# CPC / CMIIC : otagrum 0.13 (aGrUM 2) vs 0.15 (aGrUM 3)

Apprend CPC et CMIIC sur quelques golden BN et enregistre les structures apprises,
taguees par la version d'otagrum, pour isoler *ce qui change* entre les deux versions.

**Mode d'emploi**
1. Kernel `cbnsl-agrum3` -> Run All. Ecrit sous `/tmp/agrum_compare/0.15/`.
2. Kernel du venv (otagrum 0.13) -> Run All. Ecrit sous `/tmp/agrum_compare/0.13/`.
3. Reviens sur un kernel -> les cellules de comparaison lisent les deux.

L'apprentissage n'utilise pas les metriques, donc il tourne dans les deux environnements
(y compris le venv ou `StructuralMetrics` n'existe pas).

In [1]:
import sys, json
from pathlib import Path

import pyagrum as gum
import pyagrum.lib.notebook as gnb
import otagrum

from data.generators import create_default_cbn, generate_from_cbn, generate_from_sem
from pipeline.Dataset import Dataset
from pipeline.Structure import Structure
from algorithms.CPCAdapter import CPCAdapter
from algorithms.CMIICAdapter import CMIICAdapter
from experiments.gridsearch.plotting import cpdag_to_dot

VERSION = otagrum.__version__
ROOT = Path('/tmp/agrum_compare')
OUT = ROOT / VERSION
OUT.mkdir(parents=True, exist_ok=True)
print('otagrum', VERSION, '| pyagrum', gum.__version__)

/home/mathis/miniforge3/envs/cbnsl-agrum3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


otagrum 0.15 | pyagrum 3.0.0


## Golden BN

Petits exemples (v-structures elementaires) + les deux cas 10 variables du benchmark.
Chaque entree = (dag, generateur, hyperparametres).

Les hyperparametres des cas 10 variables sont ceux **stockes dans les manifestes du rapport**
(meilleure config de la grid search). Les petits exemples n'ont pas d'equivalent stocke :
CPC alpha=0.05/max=3, CMIIC alpha=0.05.

In [2]:
N_SAMPLES = 2000
SEED = 42

def make_dag(n, arcs):
    dag = gum.DAG()
    dag.addNodes(n)
    for t, h in arcs:
        dag.addArc(t, h)
    return dag

def cbn_gen(marginal='Uniform', lcc='NormalCopula'):
    def f(dag, names):
        cbn = create_default_cbn(dag, var_names=names, marginal_type=marginal, lcc_types=lcc)
        return generate_from_cbn(cbn, n_samples=N_SAMPLES, seed=SEED)
    return f

def sem_gen(noise='gaussian'):
    def f(dag, names):
        return generate_from_sem(dag, n_samples=N_SAMPLES, seed=SEED, noise_type=noise)
    return f

# no stored equivalent for the small toy graphs
TOY = {'CPC': {'alpha': 0.05, 'max_conditioning_set_size': 3}, 'CMIIC': {'alpha': 0.05}}

# hyperparameters as stored in the report's manifests (grid search best config)
P_NORMAL  = {'CPC': {'alpha': 0.1, 'max_conditioning_set_size': 2}, 'CMIIC': {'alpha': 0.01}}
P_LAPLACE = {'CPC': {'alpha': 0.1, 'max_conditioning_set_size': 3}, 'CMIIC': {'alpha': 0.01}}

DAG10 = [(1, 0), (0, 2), (2, 3), (3, 4), (3, 5), (4, 5), (4, 6), (5, 7),
         (6, 7), (6, 8), (7, 8), (9, 8), (2, 9), (9, 7)]

GOLDENS = {
    'collider':       (make_dag(3, [(0, 2), (1, 2)]), cbn_gen(), TOY),
    'chain_fork':     (make_dag(4, [(0, 1), (1, 2), (1, 3)]), cbn_gen(), TOY),
    'diamond':        (make_dag(4, [(0, 1), (0, 2), (1, 3), (2, 3)]), cbn_gen(), TOY),
    'mixed5':         (make_dag(5, [(0, 2), (1, 2), (2, 3), (3, 4), (2, 4)]), cbn_gen(), TOY),
    'cbn_normal_10':  (make_dag(10, DAG10), cbn_gen('Uniform', 'NormalCopula'), P_NORMAL),
    'sem_laplace_10': (make_dag(10, DAG10), sem_gen('laplace'), P_LAPLACE),
}

## Apprentissage + sauvegarde

In [3]:
def dump(pdag, path):
    path.write_text(json.dumps({
        'nodes': sorted(pdag.nodes()),
        'arcs':  sorted([t, h] for t, h in pdag.arcs()),
        'edges': sorted([min(u, v), max(u, v)] for u, v in pdag.edges()),
    }))

for gname, (dag, gen, params) in GOLDENS.items():
    names = [f'X{i}' for i in range(dag.size())]
    ds, golden = gen(dag, names)
    dataset = Dataset(ds.data, name=gname, feature_names=names)
    dump(golden.cpdag, OUT / f'{gname}__golden.json')

    print(f'=== {gname}  (golden arcs {sorted(dag.arcs())})')
    for aname, cls in [('CPC', CPCAdapter), ('CMIIC', CMIICAdapter)]:
        st = cls(**params[aname]).learn_structure(dataset)
        dump(st.cpdag, OUT / f'{gname}__{aname}.json')
        print(f'  {aname:6} arcs={sorted(st.cpdag.arcs())}  edges={sorted(tuple(e) for e in st.cpdag.edges())}')

=== collider  (golden arcs [(0, 2), (1, 2)])
  CPC    arcs=[]  edges=[(0, 2), (1, 2)]
  CMIIC  arcs=[]  edges=[(0, 2), (1, 2)]
=== chain_fork  (golden arcs [(0, 1), (1, 2), (1, 3)])
  CPC    arcs=[]  edges=[(0, 1), (1, 2), (1, 3)]
  CMIIC  arcs=[]  edges=[(0, 1), (1, 2), (1, 3)]
=== diamond  (golden arcs [(0, 1), (0, 2), (1, 3), (2, 3)])
  CPC    arcs=[(1, 3), (2, 3)]  edges=[(0, 1), (0, 2)]
  CMIIC  arcs=[(1, 3), (2, 3)]  edges=[(0, 1), (0, 2)]
=== mixed5  (golden arcs [(0, 2), (1, 2), (2, 3), (2, 4), (3, 4)])
  CPC    arcs=[]  edges=[(0, 2), (1, 2), (2, 3), (2, 4), (3, 4)]
  CMIIC  arcs=[(0, 2), (1, 2), (2, 3), (2, 4)]  edges=[(3, 4)]
=== cbn_normal_10  (golden arcs [(0, 2), (1, 0), (2, 3), (2, 9), (3, 4), (3, 5), (4, 5), (4, 6), (5, 7), (6, 7), (6, 8), (7, 8), (9, 7), (9, 8)])
  CPC    arcs=[(3, 4), (3, 5), (4, 6), (7, 4), (7, 5), (7, 6), (8, 4), (8, 5), (8, 6)]  edges=[(0, 1), (0, 2), (2, 3), (2, 7), (2, 8), (2, 9), (3, 7), (3, 8), (4, 5), (7, 8), (7, 9), (8, 9)]
  CMIIC  arcs=[(3,

## Affichage visuel des CPDAG appris

Pour chaque golden : golden, puis CPC et CMIIC des deux versions cote a cote.
A lancer une fois que les deux kernels ont ecrit leurs resultats.

In [4]:
def load_structure(v, name):
    d = json.loads((ROOT / v / f'{name}.json').read_text())
    pdag = gum.PDAG()
    for n in d['nodes']:
        pdag.addNodeWithId(n)
    for t, h in d['arcs']:
        pdag.addArc(t, h)
    for u, w in d['edges']:
        pdag.addEdge(u, w)
    return d, Structure(pdag)

vers = sorted(p.name for p in ROOT.iterdir() if p.is_dir())
print('versions trouvees :', vers)

for gname, (dag, gen, params) in GOLDENS.items():
    names = [f'X{i}' for i in range(dag.size())]
    dots, caps = [], []
    _, gold = load_structure(vers[0], f'{gname}__golden')
    dots.append(gnb.getDot(cpdag_to_dot(gold, names))); caps.append(f'{gname}\ngolden')
    for algo in ['CPC', 'CMIIC']:
        for v in vers:
            _, st = load_structure(v, f'{gname}__{algo}')
            dots.append(gnb.getDot(cpdag_to_dot(st, names)))
            caps.append(f'{algo}\notagrum {v}')
    gnb.sideBySide(*dots, captions=caps)

versions trouvees : ['0.13', '0.15']


<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 collider golden,<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 CPC otagrum 0.13,<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 CPC otagrum 0.15,<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 CMIIC otagrum 0.13,<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 CMIIC otagrum 0.15


<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 X2 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 chain_fork golden,<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 X2 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 CPC otagrum 0.13,<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 X2 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 CPC otagrum 0.15,<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 X2 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 CMIIC otagrum 0.13,<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 X2 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 CMIIC otagrum 0.15


<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 <!-- 2->3 --> 2->3 diamond golden,<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 1->0 --> 1->0 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 <!-- 2 --> 2 X2 <!-- 2->0 --> 2->0 <!-- 2->3 --> 2->3 CPC otagrum 0.13,<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 <!-- 2->3 --> 2->3 CPC otagrum 0.15,<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 <!-- 2->3 --> 2->3 CMIIC otagrum 0.13,<!-- 0 --> 0 X0 <!-- 1 --> 1 X1 <!-- 0->1 --> 0->1 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 3 --> 3 X3 <!-- 1->3 --> 1->3 <!-- 2->3 --> 2->3 CMIIC otagrum 0.15


<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 2->3 --> 2->3 <!-- 4 --> 4 X4 <!-- 2->4 --> 2->4 <!-- 3->4 --> 3->4 mixed5 golden,<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 3->2 --> 3->2 <!-- 4 --> 4 X4 <!-- 3->4 --> 3->4 <!-- 4->2 --> 4->2 CPC otagrum 0.13,<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 2->3 --> 2->3 <!-- 4 --> 4 X4 <!-- 2->4 --> 2->4 <!-- 3->4 --> 3->4 CPC otagrum 0.15,<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 2->3 --> 2->3 <!-- 4 --> 4 X4 <!-- 2->4 --> 2->4 <!-- 3->4 --> 3->4 CMIIC otagrum 0.13,<!-- 0 --> 0 X0 <!-- 2 --> 2 X2 <!-- 0->2 --> 0->2 <!-- 1 --> 1 X1 <!-- 1->2 --> 1->2 <!-- 3 --> 3 X3 <!-- 2->3 --> 2->3 <!-- 4 --> 4 X4 <!-- 2->4 --> 2->4 <!-- 3->4 --> 3->4 CMIIC otagrum 0.15


## Comparaison textuelle

In [5]:
assert len(vers) == 2, 'lance le notebook dans les deux kernels avant de comparer'
va, vb = vers

def raw(v, name):
    return json.loads((ROOT / v / f'{name}.json').read_text())

for gname in GOLDENS:
    print(f'### {gname}    golden arcs = {raw(va, gname + "__golden")["arcs"]}')
    for algo in ['CPC', 'CMIIC']:
        a, b = raw(va, f'{gname}__{algo}'), raw(vb, f'{gname}__{algo}')
        skel_a = {frozenset(x) for x in a['arcs']} | {frozenset(x) for x in a['edges']}
        skel_b = {frozenset(x) for x in b['arcs']} | {frozenset(x) for x in b['edges']}
        flag = ('IDENTIQUE' if a == b else
                'meme squelette, orientation differente' if skel_a == skel_b else
                'SQUELETTE DIFFERENT')
        print(f'  {algo:6} {flag}')
        if a != b:
            print(f'      {va}: arcs={a["arcs"]} edges={a["edges"]}')
            print(f'      {vb}: arcs={b["arcs"]} edges={b["edges"]}')
    print()

### collider    golden arcs = [[0, 2], [1, 2]]
  CPC    meme squelette, orientation differente
      0.13: arcs=[[0, 2], [1, 2]] edges=[]
      0.15: arcs=[] edges=[[0, 2], [1, 2]]
  CMIIC  IDENTIQUE

### chain_fork    golden arcs = []
  CPC    IDENTIQUE
  CMIIC  IDENTIQUE

### diamond    golden arcs = [[1, 3], [2, 3]]
  CPC    meme squelette, orientation differente
      0.13: arcs=[[1, 0], [2, 0]] edges=[[1, 3], [2, 3]]
      0.15: arcs=[[1, 3], [2, 3]] edges=[[0, 1], [0, 2]]
  CMIIC  IDENTIQUE

### mixed5    golden arcs = [[0, 2], [1, 2], [2, 3], [2, 4]]
  CPC    meme squelette, orientation differente
      0.13: arcs=[[0, 2], [1, 2], [3, 2], [4, 2]] edges=[[3, 4]]
      0.15: arcs=[] edges=[[0, 2], [1, 2], [2, 3], [2, 4], [3, 4]]
  CMIIC  IDENTIQUE

### cbn_normal_10    golden arcs = [[5, 7], [6, 7], [6, 8], [7, 8], [9, 7], [9, 8]]
  CPC    meme squelette, orientation differente
      0.13: arcs=[[0, 2], [3, 2], [3, 7], [3, 8], [4, 7], [4, 8], [5, 7], [5, 8], [6, 7], [6, 8], [7, 2]